# inference

> Turn a trained model into predictions: probability maps (optionally test-time augmented), label decoding, and an operating-threshold search for binary tasks.

`predict` returns per-class probabilities for both binary (sigmoid) and multi-class (softmax). `decode` collapses them to a label mask. `find_operating_threshold` sweeps the binary decision threshold to hit a target recall, which matters when a miss is far costlier than a false alarm.

In [ ]:
#| default_exp inference

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, Optional, Sequence

import numpy as np
import torch


In [ ]:
#| exporti
from segmentation_tools_pt.metrics import ObjectMetrics

## Predict probabilities

One forward pass returns `(C, H, W)` probabilities. With `tta=True` we average predictions over the four 90-degree rotations, rotating each prediction back before averaging. TTA trades compute for a small, usually-positive accuracy bump on thin structures.

In [ ]:
#| export
@torch.no_grad()
def predict(
    model,                 # trained model returning (B, C, H, W) logits
    image: torch.Tensor,   # a single (C, H, W) input
    device: torch.device,  # cpu or cuda
    tta: bool = False,     # average over 90-degree rotations
    n_augs: int = 4,       # number of rotations when tta=True
    activate: bool = True, # apply sigmoid/softmax to logits
) -> torch.Tensor:
    """Return `(C, H, W)` probabilities (sigmoid if C==1 else softmax)."""
    model.eval()
    x = image.unsqueeze(0).to(device)

    def forward(t: torch.Tensor) -> torch.Tensor:
        logits = model(t)
        if activate:
            return torch.sigmoid(logits) if logits.shape[1] == 1 else torch.softmax(logits, dim=1)
        return logits

    if not tta:
        return forward(x).squeeze(0)
    preds = []
    for k in range(n_augs):
        pr = forward(torch.rot90(x, k, dims=[2, 3]))
        preds.append(torch.rot90(pr, -k, dims=[2, 3]))
    return torch.stack(preds).mean(0).squeeze(0)

In [ ]:
#| export
def decode(
    probs: torch.Tensor,     # (C, H, W) probabilities
    threshold: float = 0.5,  # decision threshold (binary only)
) -> torch.Tensor:
    """Collapse probabilities to an `(H, W)` integer-label mask.

    Example:
        >>> decode(torch.tensor([[[0.2, 0.9]]])).tolist()
        [[0, 1]]
    """
    if probs.shape[0] == 1:
        return (probs[0] >= threshold).long()
    return probs.argmax(0)

In [ ]:
#| test
# test: shapes for binary and multi-class, with and without TTA
import torch.nn as nn
dev = torch.device('cpu')
mb = nn.Conv2d(3, 1, 3, padding=1)
p = predict(mb, torch.rand(3, 16, 16), dev)
assert p.shape == (1, 16, 16)
assert predict(mb, torch.rand(3, 16, 16), dev, tta=True).shape == (1, 16, 16)
assert decode(p).shape == (16, 16)
mc = nn.Conv2d(3, 3, 3, padding=1)
pc = predict(mc, torch.rand(3, 16, 16), dev)
assert pc.shape == (3, 16, 16) and decode(pc).shape == (16, 16)

## Operating-threshold search (binary)

Sweep the sigmoid threshold over a validation loader and report pixel recall/precision/IoU plus object-level detection at each. We recommend the **highest** threshold that still meets `target_recall`: highest precision among the options that do not start missing targets.

In [ ]:
#| export
@torch.no_grad()
def find_operating_threshold(
    model,                          # trained binary model
    val_loader,                     # validation DataLoader of {'image','mask'}
    device: torch.device,           # cpu or cuda
    target_recall: float = 0.99,    # minimum pixel recall to accept
    overlap_thresh: float = 0.5,    # object overlap fraction for detection
    thresholds: Optional[Sequence[float]] = None,  # thresholds to scan
) -> Dict:
    """Scan thresholds; return the recommended one plus the full sweep."""
    model.eval()
    thresholds = list(thresholds) if thresholds is not None else list(np.arange(0.05, 0.95, 0.05))
    prob_imgs, gt_imgs = [], []
    for batch in val_loader:
        images = batch['image'].to(device)
        p = torch.sigmoid(model(images))[:, 0].cpu().numpy()
        prob_imgs.append(p)
        gt_imgs.append((batch['mask'].numpy() > 0).astype(np.uint8))
    probs = np.concatenate(prob_imgs)        # (N, H, W)
    gts = np.concatenate(gt_imgs)            # (N, H, W)
    pf, gf = probs.reshape(-1), gts.reshape(-1).astype(bool)
    sweep = []
    for t in thresholds:
        pred = pf >= t
        tp = int((pred & gf).sum()); fp = int((pred & ~gf).sum()); fn = int((~pred & gf).sum())
        obj = ObjectMetrics(overlap_thresh)
        for pimg, gimg in zip(probs, gts):
            obj.update((pimg >= t).astype(np.uint8), gimg)
        od = obj.compute()
        sweep.append({'threshold': round(float(t), 2),
                      'recall': round(tp / (tp + fn + 1e-9), 4),
                      'precision': round(tp / (tp + fp + 1e-9), 4),
                      'iou': round(tp / (tp + fp + fn + 1e-9), 4),
                      'detection_rate': round(od['detection_rate'], 4),
                      'missed': od['missed'], 'false_alarms': od['false_alarms']})
    ok = [r for r in sweep if r['recall'] >= target_recall]
    best = max(ok, key=lambda r: r['threshold']) if ok else max(sweep, key=lambda r: r['recall'])
    return {'recommended_threshold': best['threshold'], 'achieved_recall': best['recall'],
            'achieved_precision': best['precision'], 'sweep': sweep}

In [ ]:
#| test
# test: threshold search returns a valid recommendation on a tiny val loader
from torch.utils.data import Dataset, DataLoader

class _Val(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        m = torch.zeros(16, 16, dtype=torch.long); m[4:12, 4:12] = 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': str(i)}

res = find_operating_threshold(mb, DataLoader(_Val(), batch_size=2), dev, target_recall=0.0, thresholds=[0.3, 0.5, 0.7])
assert 0.0 < res['recommended_threshold'] < 1.0 and len(res['sweep']) == 3

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()